Install libraries to connect to file on google drive

In [13]:
pip install google-api-python-client google-auth-oauthlib requests

2746.17s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 44.9 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 53.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [google-api-python-client]api-python-client]
Note: you may need to restart the kernel to use updated packages.


Now authenticate

In [8]:
from pathlib import Path
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
TOKEN = PROJECT_DIR/"token.json"

creds = None
if TOKEN.exists():
    creds = Credentials.from_authorized_user_file(TOKEN, SCOPES)
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(str(CREDENTIALS), SCOPES)
        creds = flow.run_local_server(port=0)
    TOKEN.write_text(creds.to_json())

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=9219590340-3s026ep4vdbvqsto2n2fdqqq23ovfgui.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A60744%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly&state=EF1RSRqwARyXzdifOmsKz6KqOMCqCv&code_challenge=59MWoHS8N_2S7nueDeJvpNoiJ5b1NvA2PcHcMgMWmq8&code_challenge_method=S256&access_type=offline


In [ ]:

from pathlib import Path

PROJECT_DIR = Path("/Users/ebar/Dropbox/aResearch/NoCoWild/zooniverse-panoptes")  # edit this
CREDENTIALS = PROJECT_DIR / "credentials.json"
TOKEN = PROJECT_DIR / "token.json"

assert CREDENTIALS.exists(), f"Not found: {CREDENTIALS}"


Now stream the file into pandas

In [12]:
import requests
import pandas as pd

#the file id is the long tring in the file share link
#zooniverse classification file
#https://drive.google.com/file/d/1ibs59JwqP9I0CuETUmU5_Ulcn-2EqZqf/view?usp=drive_link

FILE_ID = "1ibs59JwqP9I0CuETUmU5_Ulcn-2EqZqf"
url = f"https://www.googleapis.com/drive/v3/files/{FILE_ID}?alt=media"
headers = {"Authorization": f"Bearer {creds.token}"}

cols = ["classification_id", "user_name", "workflow_id",
        "created_at", "annotations", "subject_data", "subject_ids"]

chunks = []
with requests.get(url, headers=headers, stream=True) as r:
    r.raise_for_status()
    r.raw.decode_content = True
    for chunk in pd.read_csv(r.raw, usecols=cols, chunksize=100_000):
        chunk = chunk[chunk["workflow_id"] == 12345]
        chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(df.shape)


(0, 7)


In [15]:
df.head(10)

df

,classification_id,user_name,workflow_id,created_at,annotations,subject_data,subject_ids
